In [2]:
%cd /drive2/ryusejong/LFF
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
import json 
import time 
import re
import random
import numpy as np 
from tqdm.auto import tqdm
from util.utils import set_seed, read_data, save_result, get_answer_from_text, chat_huggingface, construct_conversation
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel

seed = 42
set_seed(seed)

/drive2/ryusejong/LFF


/drive2/ryusejong/miniconda3/envs/llm1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def find_smaller_than_T(nums, T):
    for idx, num in enumerate(nums):
        if num < T:
            return True, idx
    return False, None

In [4]:
# Parameters
T = 0.6

# Ouput file
output_path = "reward/Llama-PRM800K/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_train_512_seed42_portion0.1.jsonl"

output_file = read_data(output_path)

print(f"Output file: {len(output_file)}")

Output file: 747


In [5]:
verified_correct = []
verified_incorrect = []
not_verified_correct = []
not_verified_incorrect = []

for i in tqdm(range(len(output_file))):
    true_answer = output_file[i]["answer"]
    pred_answer = output_file[i]["pred_ans"]
    step_probs = output_file[i]["step_probs"]
    
    vf, idx = find_smaller_than_T(step_probs, T)

    if vf:
        if pred_answer == true_answer:
            verified_correct.append(output_file[i])
        else:
            verified_incorrect.append(output_file[i])
    else:
        if pred_answer == true_answer:
            not_verified_correct.append(output_file[i])
        else:
            not_verified_incorrect.append(output_file[i])

print(f"verified_correct: {len(verified_correct)}\nIndex: {[o['index'] for o in verified_correct]}\n")
print(f"verified_incorrect: {len(verified_incorrect)}\nIndex: {[o['index'] for o in verified_incorrect]}\n")
print(f"not_verified_correct: {len(not_verified_correct)}\nIndex: {[o['index'] for o in not_verified_correct]}\n")
print(f"not_verified_incorrect: {len(not_verified_incorrect)}\nIndex: {[o['index'] for o in not_verified_incorrect]}\n")
print(f"total num: {len(verified_correct) + len(verified_incorrect) + len(not_verified_correct) + len(not_verified_incorrect)}")

100%|██████████| 747/747 [00:00<00:00, 487497.29it/s]

verified_correct: 73
Index: [5, 15, 18, 32, 34, 45, 47, 48, 98, 114, 142, 150, 155, 160, 162, 163, 172, 173, 177, 201, 219, 220, 229, 237, 238, 259, 260, 276, 293, 310, 335, 340, 361, 367, 381, 383, 395, 398, 399, 429, 430, 445, 447, 448, 451, 455, 456, 462, 464, 476, 505, 514, 531, 532, 551, 558, 568, 569, 571, 583, 585, 616, 621, 629, 643, 649, 653, 664, 675, 680, 697, 728, 731]

verified_incorrect: 91
Index: [19, 21, 26, 52, 54, 64, 76, 80, 92, 94, 112, 121, 124, 137, 138, 141, 148, 151, 157, 159, 167, 183, 189, 195, 203, 205, 213, 222, 227, 246, 249, 266, 269, 277, 282, 303, 305, 312, 313, 317, 320, 325, 337, 342, 344, 349, 362, 370, 371, 376, 407, 411, 418, 423, 427, 453, 457, 461, 463, 475, 482, 489, 518, 521, 522, 530, 537, 538, 545, 555, 557, 564, 575, 590, 598, 612, 623, 624, 625, 635, 644, 648, 654, 702, 711, 714, 718, 720, 730, 734, 744]

not_verified_correct: 551
Index: [0, 1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 20, 22, 23, 24, 25, 27, 28, 30, 31, 33, 35, 36, 3

In [24]:
# Parameters
T = 0.6

# Ouput file
output_path = "output/LFF_v11/GSM8K_Llama-3-8B-Instruct_LFF_v11_1_train_512_seed42_portion0.1.jsonl"

output_file = read_data(output_path)

print(f"Output file: {len(output_file)}")

Output file: 747


In [21]:
def print_accuracy_table(a1, a2, a3, a4):
    print(f"{'verify / correct':<15} | {'correct':^6} | {'incorrect':^6} |")
    print("-" * 33)
    print(f"{'verified':<15} | {a1:^6.2f} | {a2:^6.2f} |")
    print(f"{'not verified':<15} | {a3:^6.2f} | {a4:^6.2f} |")

# 사용 예시
print_accuracy_table(0.91, 0.88, 0.94, 0.89)


verify / correct | correct | incorrect |
---------------------------------
verified        |  0.91  |  0.88  |
not verified    |  0.94  |  0.89  |


In [25]:
verified_correct = []
verified_correct_verified = []
verified_correct_not_verified = []

verified_incorrect = []
verified_incorrect_verified = []
verified_incorrect_not_verified = []

not_verified_correct = []
not_verified_correct_verified = []
not_verified_correct_not_verified = []

not_verified_incorrect = []
not_verified_incorrect_verified = []
not_verified_incorrect_not_verified = []

for i in tqdm(range(len(output_file))):
    true_answer = output_file[i]["answer"]
    pred_answer = output_file[i]["pred_ans"]
    step_probs1 = output_file[i]["step_probs1"]
    step_probs2 = output_file[i]["step_probs2"]
    
    vf1, idx1 = find_smaller_than_T(step_probs1, T)
    vf2, idx2 = find_smaller_than_T(step_probs2, T)

    if vf1:
        if pred_answer == true_answer:
            verified_correct.append(output_file[i])
            if vf2:
                verified_correct_verified.append(output_file[i])
            else: 
                verified_correct_not_verified.append(output_file[i])
        else:
            verified_incorrect.append(output_file[i])
            if vf2:
                verified_incorrect_verified.append(output_file[i])
            else:
                verified_incorrect_not_verified.append(output_file[i])
    else:
        if pred_answer == true_answer:
            not_verified_correct.append(output_file[i])
            if vf2:
                not_verified_correct_verified.append(output_file[i])
            else:
                not_verified_correct_not_verified.append(output_file[i])
        else:
            not_verified_incorrect.append(output_file[i])
            if vf2:
                not_verified_incorrect_verified.append(output_file[i])
            else:
                not_verified_incorrect_not_verified.append(output_file[i])

print(f"verified_correct: {len(verified_correct)}\nIndex: {[o['index'] for o in verified_correct]}")
print(f"\tverified_correct_verified: {len(verified_correct_verified)}\n\tIndex: {[o['index'] for o in verified_correct_verified]}")
print(f"\tverified_correct_not_verified: {len(verified_correct_not_verified)}\n\tIndex: {[o['index'] for o in verified_correct_not_verified]}\n")

print(f"verified_incorrect: {len(verified_incorrect)}\nIndex: {[o['index'] for o in verified_incorrect]}")
print(f"\tverified_incorrect_verified: {len(verified_incorrect_verified)}\n\tIndex: {[o['index'] for o in verified_incorrect_verified]}")
print(f"\tverified_incorrect_not_verified: {len(verified_incorrect_not_verified)}\n\tIndex: {[o['index'] for o in verified_incorrect_not_verified]}\n")

print(f"not_verified_correct: {len(not_verified_correct)}\nIndex: {[o['index'] for o in not_verified_correct]}")
print(f"\tnot_verified_correct_verified: {len(not_verified_correct_verified)}\n\tIndex: {[o['index'] for o in not_verified_correct_verified]}")
print(f"\tnot_verified_correct_not_verified: {len(not_verified_correct_not_verified)}\n\tIndex: {[o['index'] for o in not_verified_correct_not_verified]}\n")

print(f"not_verified_incorrect: {len(not_verified_incorrect)}\nIndex: {[o['index'] for o in not_verified_incorrect]}")
print(f"\tnot_verified_incorrect_verified: {len(not_verified_incorrect_verified)}\n\tIndex: {[o['index'] for o in not_verified_incorrect_verified]}")
print(f"\tnot_verified_incorrect_not_verified: {len(not_verified_incorrect_not_verified)}\n\tIndex: {[o['index'] for o in not_verified_incorrect_not_verified]}\n")

print(f"total num: {len(verified_correct) + len(verified_incorrect) + len(not_verified_correct) + len(not_verified_incorrect)}")

print()
print("original")
print_accuracy_table(len(verified_correct), len(verified_incorrect), len(not_verified_correct), len(not_verified_incorrect))

print()
print("modified")
print_accuracy_table(len(verified_correct_verified) + len(not_verified_correct_verified), len(verified_incorrect_verified) + len(not_verified_incorrect_verified), len(not_verified_correct_not_verified) + len(verified_correct_not_verified), len(not_verified_incorrect_not_verified) + len(verified_incorrect_not_verified))

100%|██████████| 747/747 [00:00<00:00, 369300.46it/s]

verified_correct: 73
Index: [5, 15, 18, 32, 34, 45, 47, 48, 98, 114, 142, 150, 155, 160, 162, 163, 172, 173, 177, 201, 219, 220, 229, 237, 238, 259, 260, 276, 293, 310, 335, 340, 361, 367, 381, 383, 395, 398, 399, 429, 430, 445, 447, 448, 451, 455, 456, 462, 464, 476, 505, 514, 531, 532, 551, 558, 568, 569, 571, 583, 585, 616, 621, 629, 643, 649, 653, 664, 675, 680, 697, 728, 731]
	verified_correct_verified: 68
	Index: [5, 15, 18, 32, 45, 47, 48, 98, 114, 142, 150, 155, 160, 162, 163, 172, 173, 177, 201, 219, 220, 229, 237, 238, 259, 260, 276, 293, 310, 335, 340, 361, 367, 381, 383, 395, 398, 399, 429, 430, 445, 447, 448, 451, 455, 456, 462, 464, 476, 514, 531, 532, 551, 558, 568, 569, 571, 583, 621, 643, 649, 653, 664, 675, 680, 697, 728, 731]
	verified_correct_not_verified: 5
	Index: [34, 505, 585, 616, 629]

verified_incorrect: 91
Index: [19, 21, 26, 52, 54, 64, 76, 80, 92, 94, 112, 121, 124, 137, 138, 141, 148, 151, 157, 159, 167, 183, 189, 195, 203, 205, 213, 222, 227, 246, 249, 2